<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Sensibilidad_centroides.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
eph = kagglehub.dataset_download("leonardocaravaggio/ge-images8")

100%|██████████| 304M/304M [00:05<00:00, 58.2MB/s]

Extracting files...


In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os
import timm
import torch.nn as nn

# Cargar MobileNetV2 hasta la capa 10
model = models.mobilenet_v2(pretrained=True)
model = nn.Sequential(*list(model.features[:10]))
model.eval()

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def _global_pool(feats: torch.Tensor) -> torch.Tensor:
    # Acepta [B,C,H,W] o [B,L,C]
    if feats.dim() == 4:
        # mapa -> avg pool espacial -> [B, C]
        return F.adaptive_avg_pool2d(feats, (1, 1)).squeeze(-1).squeeze(-1)
    elif feats.dim() == 3:
        # tokens -> promedio sobre L -> [B, C]
        return feats.mean(dim=1)
    else:
        raise RuntimeError(f"Forma de features no esperada: {feats.shape}")

def extract_index(image_path):
    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0)  # [1, 3, 224, 224]
    with torch.no_grad():
        feats = model(x)
        # si algún bloque devuelve una tupla/lista, tomamos el primer tensor
        if isinstance(feats, (list, tuple)):
            feats = feats[0]
        pooled = _global_pool(feats).squeeze(0)  # [C]
        inequality_index = np.std(pooled.cpu().numpy())
    return inequality_index

def compute_inequality(image_5km_path, image_10km_path):
    return {
        "Desigualdad_5km": extract_index(image_5km_path),
        "Desigualdad_10km": extract_index(image_10km_path),
    }

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 60.3MB/s]


In [12]:
import os
from tqdm import tqdm
from PIL import Image
from io import BytesIO

def pil_to_bytes(img):
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    return buf

def crop_to_square_center(img, size=1773):
    width, height = img.size
    side = min(width, height, size)
    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side
    return img.crop((left, top, right, bottom))

import os
from tqdm import tqdm
from PIL import Image
import pandas as pd

resultados_list = []

# Filtramos todas las imágenes 5K
imagenes_5k = [f for f in os.listdir(eph) if f.endswith("5K.png")]

for img_5k_name in tqdm(imagenes_5k, desc="Procesando subzonas"):
    # Nombre de la subzona (saco el " - 5K.png")
    nombre_subzona = img_5k_name.replace(" - 5K.png", "")

    img_5k_path = os.path.join(eph, img_5k_name)
    img_10k_path = os.path.join(eph, f"{nombre_subzona} - 10K.png")

    if not os.path.exists(img_10k_path):
        print(f"⚠️ No se encontró la imagen 10K para {nombre_subzona}, se saltea.")
        continue

    try:
        # Abrir y recortar imágenes
        img_5k = crop_to_square_center(Image.open(img_5k_path))
        img_10k = crop_to_square_center(Image.open(img_10k_path))

        # Convertir a objetos tipo archivo
        img_5k_io = pil_to_bytes(img_5k)
        img_10k_io = pil_to_bytes(img_10k)

        # Calcular desigualdad
        resultados = compute_inequality(img_5k_io, img_10k_io)

        # Guardar resultados en lista
        resultados_list.append({
            "Subzona": nombre_subzona,
            **resultados  # si compute_inequality devuelve un dict
        })

    except Exception as e:
        print(f"❌ Error procesando {nombre_subzona}: {e}")

# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados_list)

Procesando subzonas: 100%|██████████| 9/9 [00:30<00:00,  3.34s/it]


In [13]:
df_resultados

,Subzona,Desigualdad_5km,Desigualdad_10km
0,Gran_La_Plata_SurOeste,0.292597,0.296197
1,Gran La Plata,0.324757,0.302134
2,Gran_La_Plata_SurEste,0.284999,0.277276
3,Gran_La_Plata_Oeste,0.309255,0.269382
4,Gran_La_Plata_NorteOeste,0.285104,0.280996
5,Gran_La_Plata_Este,0.321275,0.275262
6,Gran_La_Plata_Sur,0.313138,0.283964
7,Gran_La_Plata_Norte,0.323930,0.271455
8,Gran_La_Plata_NorteEste,0.255604,0.243465


In [33]:
import numpy as np
import pandas as pd

ciudades=pd.read_csv("Gini_EPH.csv")

ciudades['Desigualdad_5km']=np.nan
ciudades['Desigualdad_10km']=np.nan

In [34]:
eph_todos = kagglehub.dataset_download("leonardocaravaggio/ge-images5")

100%|██████████| 0.99G/0.99G [00:11<00:00, 89.7MB/s]

Extracting files...


In [35]:
import os
import shutil

# Ruta de origen y destino
src_dir = eph_todos+"/Imagenes4"
dst_dir = eph_todos

# Crear lista de archivos
archivos = os.listdir(src_dir)

# Mover sin sobrescribir
for archivo in archivos:
    origen = os.path.join(src_dir, archivo)
    destino = os.path.join(dst_dir, archivo)

    if not os.path.exists(destino):  # si no existe en destino, mover
        shutil.move(origen, destino)
    else:
        print(f"⚠️ Ya existe: {archivo}, no se movió.")

print("✅ Movimiento completado.")

⚠️ Ya existe: Santiago del Estero-La Banda - 10K.png, no se movió.
⚠️ Ya existe: Gran La Plata - 1K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 5K.png, no se movió.
⚠️ Ya existe: Gran La Plata - 5K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 1K.png, no se movió.
⚠️ Ya existe: Gran La Plata - 10K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 10K.png, no se movió.
✅ Movimiento completado.


In [40]:
for i in tqdm(range(len(ciudades)), desc="Procesando ciudades"):
    nombre_aglomerado = ciudades.loc[i, "Nombre_Aglomerado"]

    img_5k_path = os.path.join(eph_todos, f"{nombre_aglomerado} - 5K.png")
    img_10k_path = os.path.join(eph_todos, f"{nombre_aglomerado} - 10K.png")


    try:
        # Abrir y recortar imágenes centradas a 1773x1773
        img_5k = crop_to_square_center(Image.open(img_5k_path))
        img_10k = crop_to_square_center(Image.open(img_10k_path))


        # Convertir imágenes a objetos tipo archivo
        img_5k_io = pil_to_bytes(img_5k)
        img_10k_io = pil_to_bytes(img_10k)


        resultados = compute_inequality(img_5k_io,img_10k_io)

        # Guardar en el DataFrame
        ciudades.loc[i, "Desigualdad_5km"] = resultados["Desigualdad_5km"]
        ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]

    except Exception as e:
        print(f"⚠️ Error en {nombre_aglomerado}: {e}")

Procesando ciudades: 100%|██████████| 32/32 [01:26<00:00,  2.69s/it]


In [41]:
import pandas as pd
import statsmodels.api as sm

# --- a) Variación interna (Gran La Plata) ---

# Filtrar subzonas de Gran La Plata
df_la_plata = df_resultados[df_resultados["Subzona"].str.contains("La_Plata")]

# Medidas de dispersión
variacion_5k = df_la_plata["Desigualdad_5km"].std()
variacion_10k = df_la_plata["Desigualdad_10km"].std()

print("Desvío estándar Gran La Plata:")
print(f"5 km: {variacion_5k:.4f}")
print(f"10 km: {variacion_10k:.4f}")


# --- b) Variación general entre ciudades (EPH) ---

# Medidas de dispersión
variacion_gini = ciudades["Gini_Hogares"].std()
print("\nDesvío estándar Gini real (todas las ciudades):", round(variacion_gini, 4))

Desigualdad_5km = ciudades["Desigualdad_5km"].std()
print("\nDesvío estándar Desigualdad 5km (todas las ciudades):", round(Desigualdad_5km, 4))

Desigualdad_10km = ciudades["Desigualdad_10km"].std()
print("\nDesvío estándar Desigualdad 10km (todas las ciudades):", round(Desigualdad_10km, 4))

Desvío estándar Gran La Plata:
5 km: 0.0231
10 km: 0.0152

Desvío estándar Gini real (todas las ciudades): 0.031

Desvío estándar Desigualdad 5km (todas las ciudades): 0.0331

Desvío estándar Desigualdad 10km (todas las ciudades): 0.0292


In [42]:
ciudades

,Unnamed: 0,AGLOMERADO,Nombre_Aglomerado,Gini_Personas,Gini_Hogares,Desigualdad_5km,Desigualdad_10km
0,0,2,Gran La Plata,0.477884,0.430569,0.324757,0.302134
1,25,32,Ciudad Autónoma de Buenos Aires,0.405170,0.415689,0.255834,0.224338
2,23,30,Santa Rosa-Toay,0.435322,0.413948,0.221171,0.203717
3,8,10,Gran Mendoza,0.414483,0.404377,0.336852,0.284709
4,26,33,Partidos del Gran Buenos Aires,0.430662,0.397060,0.240052,0.228271
5,28,36,Río Cuarto,0.450962,0.391219,0.266836,0.248631
6,24,31,Ushuaia-Río Grande,0.447865,0.384391,0.189234,0.176991
7,13,17,Neuquén-Plottier,0.384695,0.382769,0.222778,0.213567
8,18,23,Gran Salta,0.389239,0.381227,0.236264,0.243068
9,3,5,Gran Santa Fe,0.415640,0.378615,0.214201,0.209465


In [31]:
import pandas as pd
import statsmodels.api as sm

# --- Filtrar Gran La Plata ---
# Copiar df para no modificar el original
df_la_plata_filtrado = df_la_plata.copy()

# Reemplazar "Gran" por "Gran La Plata"
df_la_plata_filtrado["Ciudad"] = "Gran La Plata"

# Filtrar Gini de Gran La Plata
gini_la_plata = gini_eph[gini_eph["Ciudad"] == "Gran La Plata"][["Ciudad", "Gini_Hogares"]]

# Merge
df_merge = df_la_plata_filtrado.merge(gini_la_plata, on="Ciudad", how="left")

print(df_merge)


# --- Regresión: Gini_PHogares ~ Desigualdad_5km + Desigualdad_10km ---
X = df_merge[["Desigualdad_5km", "Desigualdad_10km"]]
X = sm.add_constant(X)  # Agregar intercepto
y = df_merge["Gini_Hogares"]

if len(df_merge) > 1:
    modelo = sm.OLS(y, X).fit()
    print("\nResumen regresión:")
    print(modelo.summary())
else:
    print("\nNo hay suficientes observaciones para correr la regresión.")


                    Subzona  Desigualdad_5km  Desigualdad_10km         Ciudad  \
0    Gran_La_Plata_SurOeste         0.292597          0.296197  Gran La Plata   
1     Gran_La_Plata_SurEste         0.284999          0.277276  Gran La Plata   
2       Gran_La_Plata_Oeste         0.309255          0.269382  Gran La Plata   
3  Gran_La_Plata_NorteOeste         0.285104          0.280996  Gran La Plata   
4        Gran_La_Plata_Este         0.321275          0.275262  Gran La Plata   
5         Gran_La_Plata_Sur         0.313138          0.283964  Gran La Plata   
6       Gran_La_Plata_Norte         0.323930          0.271455  Gran La Plata   
7   Gran_La_Plata_NorteEste         0.255604          0.243465  Gran La Plata   

   Gini_Hogares  
0      0.430569  
1      0.430569  
2      0.430569  
3      0.430569  
4      0.430569  
5      0.430569  
6      0.430569  
7      0.430569  

Resumen regresión:
                            OLS Regression Results                            
Dep. Vari

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
